# 6. Elastic Net
### Combines L1 and L2 Regularization

**Elastic Net** combines both $L_1$ (Lasso) and $L_2$ (Ridge) penalties, overcoming the limitations of each when handling correlated features or high-dimensional datasets ($p > n$).

#### Key Concepts:
1. **Loss Function**:
   $$J(w) = \frac{1}{2m} \|y - X w\|^2 + \alpha \rho \|w\|_1 + \frac{\alpha(1-\rho)}{2} \|w\|_2^2$$
2. **Parameters**:
   - `alpha`: Overall regularization strength.
   - `l1_ratio` ($ho$): Mixing parameter between $L_1$ and $L_2$.
     - $\rho = 1$: Equivalent to pure Lasso.
     - $\rho = 0$: Equivalent to pure Ridge.
3. **Grouping Effect**: Strongly correlated features tend to be selected or shrunk together rather than arbitrarily dropping one.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.metrics import r2_score, mean_squared_error


In [ ]:
# Step 2: Load Data and Standardize
diabetes = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.25, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Step 3: ElasticNetCV Tuning (Alpha & L1-Ratio Grid)
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95]
elastic_cv = ElasticNetCV(l1_ratio=l1_ratios, alphas=np.logspace(-3, 1, 30), cv=5, random_state=42)
elastic_cv.fit(X_train_scaled, y_train)

y_pred = elastic_cv.predict(X_test_scaled)
print(f"Optimal Alpha: {elastic_cv.alpha_:.4f}")
print(f"Optimal L1 Ratio: {elastic_cv.l1_ratio_:.2f}")
print(f"Elastic Net Test R²: {r2_score(y_test, y_pred):.4f}")
print(f"Elastic Net Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

coef_df = pd.DataFrame({"Feature": diabetes.feature_names, "Coefficient": elastic_cv.coef_})
print("
Elastic Net Coefficients:
", coef_df)


In [ ]:
# Step 4: Actual vs Predicted Plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, color='navy', alpha=0.7, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title(f"Elastic Net Predictions (R²: {r2_score(y_test, y_pred):.3f})")
plt.grid(True)
plt.show()


### Summary
- **Strengths**: Handles correlated feature groups smoothly, provides sparsity while maintaining numerical stability, ideal for high-dimensional genomics and tabular data.
- **Weaknesses**: Computationally more intensive due to dual hyperparameter search (`alpha` and `l1_ratio`).
